# Every subpopulation: error against adaptability

Whether a finetuned "improvement" is real adaptation or a relocated center.
Per question, model/variant and mode, scored over **every retained
subpopulation cell of the run** — all five countries, 687 cells where a run is
complete (639 for `d_polpos`, where 48 cells have no human answers): mean
prediction error `E = mean_i nEMD(WVS_i, LLM_i)`, dispersion
`D = median pairwise nEMD` within the survey cells and within the model's, the
adaptability ratio `A = D_LLM / D_WVS`, and the center relocations
`C_<country> = nEMD(mean_i WVS_i, mean_i LLM_i)` over one country's cells and
`C_pop`, the same quantity over every retained cell.

Every population quantity has a twin per **reference country** — Germany (144
cells) and Mexico (131), the two cultures the finetuned variants target — so
`e_mean_nemd_german` and `e_mean_nemd_mexican`, `adaptability_ratio_german` and
`adaptability_ratio_mexican`, and so on. Those twins are what the ticks draw: an
upright `|` for Germany, a `/` slash for Mexico. A tick is not a second run: it
is the same run scored on that country only, so the distance between a marker
and its tick is what this model does differently on that country's cells, and
the distance between the as-released variant's tick and the tuned variant's tick
is what the culture-LLM finetuning bought on the cells it was fitted for.

**A tuned variant wears only the tick of the country it targets** — `german` the
`|`, `spanish-mx` the `/` — because the other country's cells are not what it was
fitted for and a second tick there is noise. The as-released variant wears both,
since it is the baseline each tuned tick is read against; drop it and there is
nothing to measure the finetuning from.

A **variant** is one of the conditions a base model is evaluated under:
`base`, the model as released with nothing attached, or a culture-finetuned
variant such as `german`. The CSV column and the run-directory path segment
are both still spelled `arm`, so `row["arm"]` in the code and "variant" in
this prose name the same thing.

`A = 1` means the model spreads its subpopulation answers as much as the
survey's own cells do; `A -> 0` means one answer wearing 687 names. The ideal
corner is `E -> 0`, `A -> 1`.

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = as released, solid = `german`, bottom-half = `spanish-mx` |
| dotted rule + `\|` tick | not a run: the same run over the German cells alone |
| dotted rule + `/` tick | not a run: the same run over the Mexican cells alone |
| which ticks a marker wears | its own target: `german` gets the `\|`, `spanish-mx` the `/`, as-released both |
| x | E, mean nEMD to every WVS cell |
| y | A, model dispersion over survey dispersion |

Only fill separates the two variants, so nothing else may be drawn inside a
marker: an earlier draft put a white centre dot on every solid marker, which
at 6.5pt ate the fill and made the tuned variant read as a second hollow as-released marker.
The country companions were likewise second *markers* in the model's pastel
tint, a third thing shaped like a variant — the supervisor read one panel as
three as-released runs. Both are gone: fill alone says which variant, and each
country reference is a tick on a dotted rule. The two ticks are the same glyph
at different angles — `|` upright for Germany, `/` at 45 degrees for Mexico —
and neither angle is a shape any series wears, so a tick can never be misread
as a run.

Fill now carries **three** states, not two, because a model can be finetuned for
more than one culture and `german` and `spanish-mx` were previously both drawn
solid — indistinguishable inside a model's own legend block. Hollow is the model
as released, solid is `german`, and a bottom-half fill is `spanish-mx`. The
half-fill is `bottom` rather than `left`: on the triangles a left/right split is
a sliver, and the unfilled half is painted opaque white rather than left
transparent so it survives the overlaps these panels are full of.

Every dashed rule on these plates is a **fixed reference, never a fit**, which
is why it never tilts: `A = 1` on the adaptability plates, the identity
diagonal on the centers plate. They mark where a model would sit if it matched
the survey's dispersion, or sat equally far from both pools — not a trend
through the points.

A second plate puts the two relocations against each other, `x = C_<country>`
against `y = C_pop`, with the diagonal as the indifference rule: above it a
model's centre sits closer to that country's pooled respondents than to the
pooled world, below it the reverse — where an as-released variant is expected to
live. One plate is written per reference country; the country is the x axis
itself, so no tick is drawn. Germany keeps the unqualified stem
`fig_population_center_<view>_<mode>` it has always written, and Mexico is named
in its own, `fig_population_center_mexican_<view>_<mode>`.

Five views per family — every question at once, then happiness, politics,
religion and trust alone — because nEMD scales with a question's answer count,
so the four panels share a y axis but never a x scale.

Recomputed from the run distributions, not the derived CSVs. Writes
`outputs/culture/population_adaptability.csv` plus
`figures/fig_population_adaptability_<view>_{ntp,fa}` and one centers plate per
reference country, `.png` and `.pdf`. A run that
produced only full answers contributes its `fa` rows and no `ntp` rows, and is
scored on the cells its own mode retained. A companion notebook,
`population_structure.ipynb`, reads this CSV back in and asks a third,
orthogonal question over the same cells — not how much a model disperses but
whether it disperses along the real social axes — so it must be run after this
one, not instead of it.

`.venv` has no jupyter and a bare `uv run` must not be used here — it destroys
the hand-built sm_120 `llama-cpp-python` wheel. Committed without cell
outputs.

In [1]:
from culture.fidelity_baseline import manuscript_runs
from culture.population import (
    ADAPTABILITY_TABLE,
    MODES,
    REFERENCE_COUNTRIES,
    build_adaptability,
    check_adaptability,
)
from culture.population_plates import VIEWS, adaptability_plate, center_plate
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
table = build_adaptability(runs=manuscript_runs(every_replicate=True))
check_adaptability(table)
write_csv(table, ADAPTABILITY_TABLE)
print(len(table), "rows")

308 rows


## The plates

In [3]:
files = []
for mode in MODES:
    for view in VIEWS:
        files += adaptability_plate(table, mode, view)
        for country in REFERENCE_COUNTRIES:
            files += center_plate(table, mode, view, country=country)
print(len(files), "files")

60 files


## Reading the plates

A marker low on the y axis is a model answering nearly the same way for every
subpopulation it was asked about — the compression the paper reports. A marker
near `A = 1` disperses as much as the survey does, which is necessary for
real adaptation but not sufficient: `population_structure.ipynb` asks whether
that dispersion runs along the survey's own social divides or is noise of the
right size.

The `|` tick is the same run over the German cells alone, and the `/` tick the
same run over the Mexican cells. Read each against its own marker, then read a
tuned variant's tick against the as-released variant's tick of the same country:
a tuned variant whose tick sits left of it fits that country's cells more closely
than the as-released model did. A tuned variant whose *marker* moved right while
its tick moved left bought its target country at the rest of the world's
expense — the trade the centers plate then shows directly.

Because a tuned variant carries only its own country's tick, the as-released
marker is the one place both appear together, and the gap between its two ticks
is the head start: how much better or worse this model already did on Germany
than on Mexico before any finetuning.

On the centers plate, above the diagonal is a model whose pooled answer sits
closer to that plate's pooled country than to the pooled world. An as-released
variant is expected below it. A tuned variant that crossed the diagonal relocated
its centre, which is the cheap way to look adapted: the adaptability plate is
where you check whether the dispersion followed.

The p-values in the structure companion are descriptive only — pairwise
distances are not independent — and nothing here is a significance claim.